# GPT-2 (124M), the parts that are ideas

This is the *Practice* step of `unit_10_gpt2_reproduce.md`. Do the Cold Attempt there first.

This lecture is four hours long and most of it is engineering. You do **not** rebuild it.
You write only the components where the ideas live; everything else is handed over working.
Each milestone is one cell of stubs followed by a grader cell. The grader stops at your first
failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open build-nanogpt or nanoGPT.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *PyTorch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Everything here runs on CPU in seconds: a 2-layer, 32-wide GPT on character-level Shakespeare.
The ideas do not care about the scale.

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from test_gpt2_reproduce import grade

torch.manual_seed(1337)

## Given: data and config

Character-level tokens stand in for tiktoken (same shape of object: a 1-D stream of ints).
`GPTConfig` is the lecture's dataclass at toy size.

In [ ]:
with open("../data/tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)
tokens = torch.tensor(encode(text), dtype=torch.long)   # the whole stream, ~1.1M tokens
print(f"{len(tokens):,} tokens, vocab {len(chars)}")


@dataclass
class GPTConfig:
    block_size: int = 32     # max sequence length   (GPT-2: 1024)
    vocab_size: int = 65     # number of tokens      (GPT-2: 50257)
    n_layer: int = 2         # number of Blocks      (GPT-2: 12)
    n_head: int = 2          # attention heads       (GPT-2: 12)
    n_embd: int = 32         # residual stream width (GPT-2: 768)

## Milestone 1 — CausalSelfAttention, the GPT-2 way

You wrote multi-head attention in lecture 7 as a list of `Head` modules concatenated.
GPT-2 does the same computation with **no per-head modules**: one fused `Linear` produces
q, k and v for every head at once, and the heads are separated by reshaping, not by looping.

The contract below is what the OpenAI checkpoint's parameter names force on you.
Match it exactly; the grader builds a per-head reference from *your* `c_attn` and `c_proj`.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with a fused qkv projection.

    Submodules (names are the contract; they mirror the OpenAI/HF checkpoint):
      c_attn   nn.Linear(n_embd, 3 * n_embd)  -- one matmul gives q, k, v for all heads
      c_proj   nn.Linear(n_embd, n_embd)      -- projects the merged heads back into the residual stream
    Both have biases.

    forward(x): x is (B, T, C) with C == n_embd, T <= block_size. Returns (B, T, C).
      Inside, each head works on a (B, n_head, T, head_size) view where head_size = C // n_head.
      Attention scores are scaled by 1/sqrt(head_size). The future is masked so position t
      can only see positions <= t. Heads are merged back into one C-wide vector per token
      before c_proj.
    """

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, upto=1)

## Given: MLP and Block

Straight from lecture 7, in `nn.Module` form, with GPT-2's names and its tanh-approximate GELU.
Nothing to write here; run the cell.

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # pre-norm; the residual path is a clean highway
        x = x + self.mlp(self.ln_2(x))
        return x

## Milestone 2 — DataLoaderLite

A token stream is one long 1-D tensor. Training wants `(B, T)` inputs and `(B, T)` targets.
Write the smallest thing that walks the stream and hands out batches, exactly as the lecture does.

In [ ]:
class DataLoaderLite:
    """Walks a 1-D LongTensor of tokens in order, one (B, T) batch at a time.

    tokens: 1-D LongTensor, the whole stream.
    next_batch() -> (x, y), both (B, T) LongTensors, y[i, j] is the token that follows x[i, j].
      Each call consumes the next B*T+1 tokens of the stream (why +1?) and advances by B*T.
      When the NEXT window would run off the end of the stream, go back to the start.
    """

    def __init__(self, tokens, B, T):
        raise NotImplementedError

    def next_batch(self):
        raise NotImplementedError

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, DataLoaderLite=DataLoaderLite, upto=2)

## Milestone 3 — weight tying and GPT-2 initialization

The GPT container and its forward pass are given: embeddings in, blocks, final LayerNorm,
`lm_head` out, cross-entropy if targets are supplied.

Two things are yours, both discovered in the lecture by staring at the checkpoint:
1. `tie_weights`: the token embedding and the output head are the *same* matrix.
2. `init_weights`: GPT-2's init is not PyTorch's default. Every Linear and Embedding is
   normal with std 0.02, biases zero, LayerNorm untouched — except the Linears that write
   into the residual stream, whose std shrinks with the number of such writes.

The grader measures the std of every weight group and checks the loss at init.

In [ ]:
class GPT(nn.Module):
    """GPT-2 in miniature. Given: container + forward. Yours: tie_weights, init_weights."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),   # token embedding
            wpe=nn.Embedding(config.block_size, config.n_embd),   # position embedding
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.tie_weights()
        self.init_weights()

    def tie_weights(self):
        """Make transformer.wte.weight and lm_head.weight ONE Parameter (not a copy).
        After this, model.parameters() must list that tensor exactly once."""
        raise NotImplementedError

    def init_weights(self):
        """GPT-2 init, applied to every submodule (self.named_modules() gives (name, module) pairs):
          nn.Linear     weight ~ N(0, 0.02), bias = 0
                        ...but a Linear whose name ends in 'c_proj' (attn.c_proj, mlp.c_proj:
                        the ones that add into the residual stream) gets std 0.02 * (2 * n_layer) ** -0.5
          nn.Embedding  weight ~ N(0, 0.02)
          nn.LayerNorm  leave at PyTorch's default (weight 1, bias 0)
        Use torch.nn.init.normal_ / zeros_ so it happens in place."""
        raise NotImplementedError

    def forward(self, idx, targets=None):
        """idx: (B, T) token ids. Returns (logits (B, T, vocab), loss or None)."""
        B, T = idx.size()
        assert T <= self.config.block_size, f"sequence of length {T} > block_size {self.config.block_size}"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
model = GPT(GPTConfig())
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
x, y = DataLoaderLite(tokens, 4, 32).next_batch()
logits, loss = model(x, y)
print(logits.shape, f"loss at init {loss.item():.3f}, ln(65) = {math.log(65):.3f}")

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, DataLoaderLite=DataLoaderLite, GPT=GPT, upto=3)

## Engineering — watch, don't rebuild

Everything below made the lecture's run go from 1000 ms/step to 90 ms/step. None of it
changes what the model computes. Watch these sections; nothing here is scaffolded.

| Timestamp | What | Why it is engineering, not an idea |
|---|---|---|
| 1:22:18 | GPUs, mixed precision | which floats the hardware multiplies fastest |
| 1:28:14 | Tensor Cores, TF32 | 10-bit mantissa matmuls, free 3x |
| 1:39:38 | bfloat16 autocast | same exponent range as fp32, fewer mantissa bits |
| 1:48:15 | `torch.compile` | kernel fusion; removes Python and memory round-trips |
| 2:00:18 | flash attention | never materializes the T×T matrix; same math |
| 2:06:54 | vocab 50257 → 50304 | pad to a multiple of 64 so kernels hit their fast path |
| 2:46:52 | DDP | one process per GPU, average grads with all-reduce |
| 3:10:21 | FineWeb-EDU, sharding | data plumbing at 10B tokens |

One question to carry while watching: for each row, *would the loss curve be different if you
skipped it?* If the answer is "no, just slower", it's engineering.

## Milestone 4 — gradient accumulation

GPT-3 small trained on 0.5M tokens per optimizer step. No GPU holds that batch. The lecture's
answer: several **micro-batches** whose gradients add up to *exactly* the gradient of the one
big batch you cannot afford.

**grad_accum_step(model, loader, grad_accum_steps)**
- Uses `model` and `loader.next_batch()` and nothing else. Never touches an optimizer.
- Consumes exactly `N = grad_accum_steps` batches from `loader`, one `next_batch()` call per
  micro-step: `(x_1, y_1) … (x_N, y_N)`.
- After it returns, every `p.grad` equals the gradient of
  `cross_entropy(model(cat(x_1..x_N, dim=0)), cat(y_1..y_N, dim=0))`: one backward over all
  `N*B*T` tokens at once. Whatever was in `.grad` before the call does not survive it.
- Returns the loss of that whole big batch, the mean cross-entropy over all `N*B*T` tokens, as a
  detached scalar tensor or a float. Never `None`. This is the number you would print for the step.

The grader plants `7.0` in every `.grad` before calling you, then checks, in order: returned loss
is not `None` → every parameter has a `.grad` → the loader advanced exactly `N` batches →
accumulated `.grad` equals the one-big-batch gradient (it will say if what you left equals the
last micro-batch alone, is `N`× too large, or still contains the planted `7.0`) → returned loss
equals the big-batch loss.


In [ ]:
def grad_accum_step(model, loader, grad_accum_steps):
    """Accumulate the gradient of ONE big batch made of `grad_accum_steps` micro-batches.

    Starts from cleared gradients (model.zero_grad(set_to_none=True)).
    Pulls exactly one (x, y) from loader.next_batch() per micro-step, runs model(x, y),
    and backpropagates so that afterwards every p.grad equals what a single backward pass over
    all the micro-batches' tokens (concatenated along B) would have produced.

    Returns the loss of that whole big batch as a detached scalar tensor (or float),
    i.e. the number you would print for this step.
    Does NOT touch an optimizer.
    """
    raise NotImplementedError

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, DataLoaderLite=DataLoaderLite, GPT=GPT,
      grad_accum_step=grad_accum_step, upto=4)

## Milestone 5 — the learning-rate schedule, and it learns

GPT-3's schedule: linear warmup, then cosine decay down to a floor. Write the function, run the
given loop (your attention, loader, init, accumulation and schedule wired together), then grade.

**get_lr(it, max_lr, min_lr, warmup_steps, max_steps)**
- A pure function of five numbers; `it` is the 0-indexed optimizer step. Returns a float.
- `it < warmup_steps`: `max_lr * (it + 1) / warmup_steps`. So `get_lr(0)` is one warmup step's
  worth, never `0`, and `get_lr(warmup_steps - 1)` is exactly `max_lr`.
- `warmup_steps <= it <= max_steps`: a half-cosine from `max_lr` down to `min_lr`,
  `min_lr + 0.5 * (1 + cos(pi * r)) * (max_lr - min_lr)` with `r` running from `0` at
  `warmup_steps` to `1` at `max_steps`. At the midpoint of the decay it is exactly
  `(max_lr + min_lr) / 2`; at `it == max_steps` it is exactly `min_lr`.
- `it > max_steps`: `min_lr`.

The grader checks, in order: `get_lr(0)` is not `None` → is `> 0` → equals `max_lr / warmup_steps`
→ `get_lr(warmup_steps - 1) == max_lr` → midpoint of the decay is `(max_lr + min_lr) / 2` → at and
after `max_steps` it is `min_lr` → warmup is strictly increasing → decay never increases → every
step from `0` to `max_steps + 4` matches the reference curve → then it trains a fresh tiny GPT for
300 steps with **your** `DataLoaderLite`, `GPT`, `grad_accum_step` and `get_lr` and requires the
held-out loss to end below `2.95` → and the first loss to start near `ln(vocab)`.


In [ ]:
def get_lr(it, max_lr, min_lr, warmup_steps, max_steps):
    """Learning rate for optimizer step `it` (0-indexed).

    it < warmup_steps        linear warmup from max_lr / warmup_steps (step 0) up to max_lr
                             (step warmup_steps - 1). Step 0 is not zero: a zero-lr step is wasted.
    warmup_steps <= it <= max_steps
                             cosine decay from max_lr down to min_lr, a half-cosine:
                             at the midpoint the lr is exactly (max_lr + min_lr) / 2.
    it > max_steps           min_lr.
    """
    raise NotImplementedError

In [ ]:
# Given: the training loop scaffold. Your pieces, wired together. ~3 s on CPU.
torch.manual_seed(1337)
config = GPTConfig()
model = GPT(config)
loader = DataLoaderLite(tokens, B=8, T=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, betas=(0.9, 0.95))
max_steps, warmup_steps, grad_accum_steps = 300, 20, 2

for step in range(max_steps):
    lr = get_lr(step, 3e-3, 3e-4, warmup_steps, max_steps)
    for group in optimizer.param_groups:
        group["lr"] = lr
    loss = grad_accum_step(model, loader, grad_accum_steps)
    optimizer.step()
    if step % 30 == 0 or step == max_steps - 1:
        print(f"step {step:4d} | lr {lr:.2e} | loss {float(loss):.4f}")

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, DataLoaderLite=DataLoaderLite, GPT=GPT,
      grad_accum_step=grad_accum_step, get_lr=get_lr, upto=5)

## Milestone 6 — *stretch:* AdamW param groups and where clipping goes

Two small things from the hyperparameter section. The grader uses plain SGD for the second so
the parameter update is transparent.

**configure_optimizers(model, weight_decay, learning_rate)**
- Returns a `torch.optim.AdamW` with `lr=learning_rate`, `betas=(0.9, 0.95)`, `eps=1e-8` and
  exactly two param groups:
  - every parameter with `p.dim() >= 2` (weight matrices, embedding tables), `weight_decay=weight_decay`
  - every parameter with `p.dim() < 2` (biases, LayerNorm weight and bias), `weight_decay=0.0`
- Every parameter is in exactly one group. The tied `wte`/`lm_head` tensor appears once (it is one
  `Parameter`; `model.parameters()` already yields it once).

**train_step(model, optimizer, loader, grad_accum_steps, lr, max_norm=1.0)**
- One full optimizer step built on `grad_accum_step`.
- Returns `(loss, norm)`: `loss` is what `grad_accum_step` returned; `norm` is the global L2 norm
  of the *whole accumulated* gradient over `model.parameters()`, measured **before** clipping
  (a scalar tensor; `torch.nn.utils.clip_grad_norm_` returns exactly that).
- After it returns: the gradient norm is `<= max_norm`; every `optimizer.param_groups[i]["lr"]`
  is `lr`; and every parameter has moved by exactly `-lr * clipped_grad` (under SGD), i.e. the
  optimizer stepped with the clipped gradient, not the raw one.

The grader checks, in order: returns an `AdamW` → two groups → their `weight_decay` values are
`{weight_decay, 0.0}` → the decayed group's parameter count → the no-decay group's count → every
parameter in exactly one group → `lr` / `betas` / `eps` on each group → `train_step` returns a
2-tuple → reported `norm` equals the norm of the whole accumulated gradient → the gradient was
actually clipped → each parameter moved by `-lr * clipped_grad` → `lr` was set on the groups.

**Skipping this milestone.** The final grade cell passes `skip=(6,)` so the whole grader runs
green without it. Nothing later needs `configure_optimizers` or `train_step`: the milestone-5 loop
uses a plain `AdamW` with no clipping, milestone 7 has no code, and `generate` needs only the
model. When you write this milestone, remove `skip=(6,)` from the grade cell below.


In [ ]:
def configure_optimizers(model, weight_decay, learning_rate):
    """Return torch.optim.AdamW over model.parameters() with two param groups:
      - every parameter tensor with dim >= 2 (matrices: weights, embeddings)  -> weight_decay
      - every parameter tensor with dim <  2 (vectors: biases, LayerNorm)     -> weight_decay 0.0
    betas=(0.9, 0.95), eps=1e-8, lr=learning_rate. Every parameter in exactly one group;
    the tied wte/lm_head tensor must appear once.
    """
    raise NotImplementedError


def train_step(model, optimizer, loader, grad_accum_steps, lr, max_norm=1.0):
    """One optimizer step: accumulate (grad_accum_step), clip the global gradient norm to
    max_norm, set lr on every param group, optimizer.step().

    Returns (loss, norm): the accumulated loss and the gradient norm as measured BEFORE
    clipping (torch.nn.utils.clip_grad_norm_ returns exactly that).
    """
    raise NotImplementedError

In [ ]:
grade(CausalSelfAttention=CausalSelfAttention, DataLoaderLite=DataLoaderLite, GPT=GPT,
      grad_accum_step=grad_accum_step, get_lr=get_lr,
      configure_optimizers=configure_optimizers, train_step=train_step, upto=6, skip=(6,))

## Milestone 7 — *stretch:* HellaSwag, predict the outcome (no code)

HellaSwag hands the model a context and four candidate endings; the model "picks" the ending
whose tokens it finds least surprising. The lecture scores each candidate by feeding
`context + ending` through the model and averaging the per-token loss **over the ending tokens
only**, then choosing the lowest.

Answer in the coaching chat, from reasoning, before checking anything:

1. Why average per-token loss over the ending rather than **sum** it? Predict which endings
   a sum-based scorer would systematically prefer.
2. Why exclude the context tokens from the score? What would including them change,
   given that all four candidates share the same context?
3. A 124M model has never been trained on multiple-choice questions. What about
   next-token prediction makes this evaluation meaningful anyway?
4. Predict: a model that scores well here at 10B tokens of training — would it keep
   improving on HellaSwag if you kept training on the same data for a second epoch?
   Say what you'd watch to tell.

The chat will not confirm or correct until all four are answered.

## Output — the sampling loop, from memory

Write `generate` without looking anything up: take a prefix, repeat `max_new_tokens` times:
run the model on the last `block_size` tokens, take the logits at the final position,
softmax, keep the top-k, sample one, append. Then sample a few Shakespeare lines from the
model you trained in milestone 5.

In [ ]:
@torch.no_grad()
def generate(model, prefix, max_new_tokens=100, top_k=10):
    ...


model.eval()
print(decode(generate(model, encode("ROMEO:"), 200)))
model.train()

## Scratch

Space to poke at things. `model.transformer.h[0].attn.c_attn.weight.std()` is a good first poke.